# 06 — Combined Built-in + MCP Tools

Shows how to combine **built-in tools** (Calculator, Clock) with **MCP tools** (filesystem)  
and run a multi-step agent loop that uses both.

**Prerequisites**: `npx` on PATH and `OPENAI_API_KEY` set.

In [ ]:
import json

from ravi.integrations.mcp import MCPClient, MCPTool
from ravi.extensions.tools.builtin_tools import CalculatorTool, GetCurrentTimeTool
from ravi.integrations.llm.factory import create_model_client
from ravi.kernel.messages.client_messages import UserMessage, SystemMessage, ToolExecutionResultMessage
from ravi.kernel.messages.content import TextBlock

from ravi.configs.settings import settings

CHAT_MODEL = settings.CHAT_MODEL
API_KEYS = {
    "openai":     settings.OPENAI_API_KEY,
    "anthropic":  settings.ANTHROPIC_API_KEY,
    "google":     settings.GEMINI_API_KEY,
    "groq":       settings.GROQ_API_KEY,
    "openrouter": settings.OPENROUTER_API_KEY,
}

In [ ]:
SSE_URL = "http://localhost:9000/sse"  # MCP server: docker compose -f docker/docker-compose.yml --profile mcp up -d mcp-server

async def demo():
    builtin_tools = [CalculatorTool(), GetCurrentTimeTool()]
    print(f"Built-in tools: {[t.name for t in builtin_tools]}")

    mcp_client = MCPClient()
    try:
        await mcp_client.connect_sse(SSE_URL)
        mcp_tools = await mcp_client.discover_tools()
        print(f"MCP tools: {[t.name for t in mcp_tools]}")

        all_tools = builtin_tools + mcp_tools
        print(f"Total: {len(all_tools)} tools")

        client = create_model_client(CHAT_MODEL, api_keys=API_KEYS)
        messages = [
            SystemMessage(content="You have calculator, clock, and filesystem tools. Use them."),
            UserMessage(content=[TextBlock(text="Calculate 123 * 456 and save the result to /tmp/calculation.txt")]),
        ]

        for i in range(5):
            print(f"\n--- Iteration {i+1} ---")
            response = await client.generate(
                messages=messages,
                tools=[t.get_openai_schema() for t in all_tools],
            )
            if not response.tool_calls:
                print(f"Configured chat model: {CHAT_MODEL}")
                print(f"Final answer: {response.content}")
                break
            messages.append(response)
            for tc in response.tool_calls:
                name = tc.name
                args = tc.arguments if isinstance(tc.arguments, dict) else json.loads(tc.arguments)
                print(f"Calling: {name}({args})")
                tool = next((t for t in all_tools if t.name == name), None)
                if tool:
                    res = await tool.run(**args)
                    tool_msg = ToolExecutionResultMessage.from_tool_result(
                        tool_result=res,
                        tool_call_id=tc.tool_call_id,
                        tool_name=name,
                    )
                    preview = tool_msg.content[0].text if tool_msg.content else str(res.content)
                    print(f"Result: {preview[:100]}...")
                    messages.append(tool_msg)

    except Exception as e:
        print(f"Error: {e}")
        print("  Requires: Node.js + npx")
    finally:
        if mcp_client.is_connected:
            await mcp_client.disconnect()

await demo()

Built-in tools: ['calculator', 'get_current_time']
MCP tools: ['add', 'subtract', 'multiply', 'echo', 'to_uppercase', 'word_count', 'server_info']
Total: 9 tools

--- Iteration 1 ---
Calling: calculator({'expression': '123 * 456'})
Result: [{'type': 'text', 'text': '{"result": 56088, "expression": "123 * 456"}'}]...

--- Iteration 2 ---
Calling: echo({'message': '56088'})
Result: [{'type': 'text', 'text': '56088'}]...
Calling: server_info({})
Result: [{'type': 'text', 'text': '{"name":"agent-framework-demo","transport":"sse","tools":["add","subtract...

--- Iteration 3 ---
Calling: echo({'message': 'The result of 123 * 456 is 56088. Now saving to /tmp/calculation.txt.'})
Result: [{'type': 'text', 'text': 'The result of 123 * 456 is 56088. Now saving to /tmp/calculation.txt.'}]...

--- Iteration 4 ---
Final answer: ['I don\'t have direct access to a filesystem to save files. You can manually save the result "56088" to `/tmp/calculation.txt` on your system.']


---
## API update notes (Sprints 2 & 5)

Same as notebooks 04 and 05:
- Import path: `ravi.extensions.mcp` → `ravi.integrations.mcp`
- `MCPTool.from_mcp_client(client)` → `client.discover_tools()`
- `tc.id` / `msg.id` → `tc.tool_call_id` / `msg.tool_call_id`

The combined catalog pattern (built-in + MCP tools) now works best with
`MCPCatalogAdapter` — see notebook **23_mcp_catalog_adapter** for the full flow.
